In [ ]:
import torch
import numpy as np


In [ ]:

import torch_optimizer as optim_
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
import copy
import tqdm

from scipy.optimize import minimize
import torch
import numpy as np
import copy


import torch.optim as optim


class pQED:

    def __init__(self, E_array, mu_array, LR = 0, num_epochs = 0):
        self.E_array = E_array
        self.mu_array = mu_array
        self.LR = 0
        self.num_epochs = 1


    class CoherentStateTransform(nn.Module):

        def __init__(self, n_el, n_ph, omega, lambda_vector, E_array, mu_array, coherent_state= False,):
            super().__init__()
            self.n_el = n_el
            self.n_ph = n_ph
            self.coherent_state = coherent_state
            self.omega = torch.tensor(omega, dtype=torch.double, requires_grad=True)
            self.lambda_vector = torch.tensor(lambda_vector, dtype=torch.double, requires_grad=True)
            self.E_array = torch.tensor(E_array[:n_el], dtype=torch.double, requires_grad=True)
            self.mu_array = torch.tensor(mu_array,  dtype=torch.double,requires_grad=True)


            self._d = torch.tensor(self.build_d_array(n_el, self.lambda_vector, self.mu_array, coherent_state= True), dtype=torch.double, requires_grad=True)

            # # Coherent state parameters for each state
            self.eta_k = nn.Parameter(torch.ones(n_el, dtype=torch.double) *torch.diag(self._d), requires_grad=True)
            # self.f = nn.Parameter(torch.ones(1, dtype=torch.double)  * 0.75, requires_grad=True)

        def forward(self):
            #return only eigvals
            vals, vecs , H =  self.PCQED_Hamiltonian_transformed()
            self.H = H
            return vals


        def PCQED_Hamiltonian_transformed(self):
            """
            Build the PF Hamiltonian for a system with n_el electronic states and n_ph photon states.
            """

            """
            Given an array of n_el E_R values and an n_ph states with fundamental energy omega
            build the PF Hamiltonian

            n_el : int
                the number of electronic states (n_el = 1 means only ground-state)

            n_ph : int
                the number of photon occupation states (n_ph = 1 means only the |0> state)

            omega : float
                the photon frequency

            lambda_vector : numpy array of floats
                the lambda vector

            E_array : n_el np.array of floats
                the electronic energies

            mu_array : (n_el x n_el x 3) np.array of floats
                mu[i, j, k] is the kth cartesian component of the dipole moment expectation value between
                state i and state j


            """

            self.eta_k_matrix  = torch.diag(self.eta_k)

            # Identity matrices for each subsystem
            I_matter = torch.eye(self.n_el)
            I_photon = torch.eye(self.n_ph)

            _d =  self.build_d_array(self.n_el, self.lambda_vector, self.mu_array, coherent_state=self.coherent_state)

            #create bosonic subspace operators
            b = self.create_annihilation_operator(self.n_ph)
            b_dagger = self.create_creation_operator(self.n_ph)

            E_matter = self.E_array* I_matter
            E = torch.kron( I_photon, E_matter)

            Cav =  self.omega * torch.kron((b_dagger @ b), I_matter)

            BLC_matter = _d.clone()
            BLC_photon = b_dagger + b

            # Tensor product for combined subspace
            BLC =  -torch.sqrt(self.omega/2) * torch.kron(BLC_photon, BLC_matter)

            DSE_matter  =  ( _d @ _d)
            DSE = 0.5 * torch.kron(I_photon, DSE_matter)


            H = E + Cav + DSE  +BLC

            vals, vecs  = torch.linalg.eigh(H)

            return vals, vecs, H



        def return_params(self):
            return [self.eta_k]

        
        def build_d_array(
            self,
            n_el,
            lambda_vector,
            mu_array,
            coherent_state=False,
            coherent_state_pos = None,
            coherent_state_val = None
            ):
            """
            method to compute the array d = \lambda \cdot \mu if coherent_state==False
            or d = \lambda \cdot (\mu - <\mu>) if coherent_state == True
            and store to attribute self.d_array
            """

            if coherent_state == False:
                d_array = torch.einsum(
                    "k,ijk->ij", lambda_vector, mu_array[:n_el, :n_el, :]
                )
                #print("regular; ", d_array)
            else:

                #print("cohrent satest ")
                _I = torch.eye(n_el)
                d_array = torch.einsum(
                    "k,ijk->ij", lambda_vector, mu_array[:n_el, :n_el, :]
                )

                if coherent_state_val == None and coherent_state_pos == None:
                    _d_exp = d_array[0, 0]
                elif coherent_state_val == None and coherent_state_pos != None:
                    _d_exp = d_array[coherent_state_pos, coherent_state_pos]
                else:
                    _d_exp = coherent_state_val

                d_array = d_array - _I * _d_exp
                #print("cohrent satest :",  d_array)


                

            return d_array

        
        def create_annihilation_operator(self, N):
            """
            Creates the matrix representation of the annihilation operator (b) for a harmonic oscillator
            in a Hilbert space with N levels.
            
            Parameters:
            N (int): Number of levels.
            
            Returns:
            np.ndarray: The matrix representation of the annihilation operator.
            """
            b = torch.zeros((N, N))
            
            for j in range(1, N):
                b[j-1, j] = torch.sqrt(torch.tensor(j))
            
            return b

        def create_creation_operator(self, N):
            """
            Creates the matrix representation of the creation operator (b†) for a harmonic oscillator
            in a Hilbert space with N levels.
            
            Parameters:
            N (int): Number of levels.
            
            Returns:
            np.ndarray: The matrix representation of the creation operator.
            """
            b_dagger = torch.zeros((N, N))
            
            for j in range(1, N):
                b_dagger[j, j-1] = torch.sqrt(torch.tensor(j))
            
            return b_dagger



        def create_number_operator(self, N):
            """
            Creates the matrix representation of the number operator (n) = b† * b.
            
            Parameters:
            N (int): Number of levels.
            
            Returns:
            np.ndarray: The matrix representation of the number operator.
            """
            b = self.create_annihilation_operator(N)
            b_dagger = self.create_creation_operator(N)
            
            # The number operator is n = b† * b
            n = b_dagger @ b
            
            return n

        def factorial(self, n):
            """Compute the factorial of n using PyTorch."""
            # Create a tensor of values [1, 2, ..., n] and compute the product
            return torch.cumprod(torch.arange(1, n + 1, dtype=torch.float32), dim=0)[-1]


    # Define a function to clamp the parameters
    def clamp_parameters(self, model, min_val, max_val):
        for param in model.parameters():
            param.data.clamp_(min_val, max_val)


    def PQED_Hamiltonian(  self, n_el,n_ph,omega,lambda_vector, number_to_minimize = 1, verbose =  False, coherent_state= False):

        self.model = self.CoherentStateTransform( n_el, n_ph, omega, lambda_vector, self.E_array, self.mu_array, coherent_state=coherent_state )


        #number of epochs
        n_epochs =  self.num_epochs

        #learning rate
        LR = self.LR

        #LBFGS optimizer
        optimizer = optim.LBFGS(self.model.return_params(), lr=LR)
        #Adam optimizer
        optimizer = optim.Adadelta(self.model.return_params(), lr=LR, rho = 0.96)
        # optimizer = optim.SGD(self.model.return_params(), lr=LR)
        # optimizer = torch.optim.RMSprop(self.model.parameters(), lr=LR)

        # Hold the best model
        best_loss = np.inf  # init to infinity
        best_weights = None
        history = []

        energies = []

        for epoch in range(n_epochs):
            running_loss = 0
            self.model.train()

            def closure():
                optimizer.zero_grad()
                y_pred = self.model()
                loss = torch.sum(y_pred[:number_to_minimize])
                #loss.backward(retain_graph=True)

                # Use autograd.grad instead of backward
                grads = torch.autograd.grad(loss, self.model.parameters(), create_graph=True,  allow_unused=True)
                
                # Manually set gradients for optimizer step
                for param, grad in zip(self.model.parameters(), grads):
                    param.grad = grad

                return loss


            optimizer.step(closure)
            loss = closure()
            running_loss += loss.item()

            # Clamp the parameters after the update
            self.clamp_parameters(self.model, min_val=-1.0, max_val=1.0)

            energies.append(self.model().detach().numpy())

            print("epoch-", epoch, "   loss_train------", loss.item())
            history.append(np.log(loss.detach().numpy()))  # Detach before converting to NumPy
            if loss < best_loss:
                best_loss = loss
                best_weights = copy.deepcopy(self.model.state_dict())


        # restore model and return best accuracy
        self.model.load_state_dict(best_weights)

        return np.array(energies)



    def eval(self):
        with torch.no_grad():
            vals, vecs, H = self.model.PCQED_Hamiltonian_transformed()

        return vals, vecs, H

In [ ]:
N_el = 50
N_ph = 2
omega = 0.12068
lambda_vector = np.array([0.0,0.0,0.05])


# !!! Change this to the correct path on your computer!
npy_folder = "/Users/proden/Code/POLARITONPERTURBATIONTHEORY/data/"

# these file names should still be good
E_npy_file = npy_folder + "LiH_r_scan_6311g_fci_tight_davidson_Energies.npy"
Mu_npy_file = npy_folder + "LiH_r_scan_6311g_fci_tight_davidson_Dipoles.npy"

# store energy eigenvalues in E_array
E_array = np.load(E_npy_file)
# store dipole matrix elements in Mu_array
Mu_array = np.load(Mu_npy_file)

# print their shape so we know how many elements we have
print(np.shape(E_array))
print(np.shape(Mu_array))
# print(E_array)

print(E_array[: 5, :].T.tolist()   )
print( (E_array[:5, :].T + omega).tolist())

import matplotlib.pyplot as plt
plt.plot(   np.array(E_array[: 5, :].tolist() +  (E_array[:5, :] + omega).tolist() ).T  , linestyle = 'dotted' )


N_ph = 2
omega= 0.12068

vals_for_plot = []
for i in range(0, E_array.shape[1]):
    e_ar = E_array[:, i]
    mu_ar = Mu_array[:,:,:, i]
    pqed = pQED(e_ar, mu_ar)
    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
    vals_for_plot.append(vals.detach().numpy())

#plt.plot(np.array(vals_for_plot)[:, :5])

N_ph = 5

vals_for_plot = []
for i in range(0, E_array.shape[1]):
    e_ar = E_array[:, i]
    mu_ar = Mu_array[:,:,:, i]
    pqed = pQED(e_ar, mu_ar)
    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
    vals_for_plot.append(vals.detach().numpy())

plt.plot(np.array(vals_for_plot)[:, :5])



#only get the first slice for now
E_array = E_array[:, 15]
Mu_array = Mu_array[:,:,:, 15]

In [ ]:
N_el = 50
N_ph = 2
omega = 0.12068
lambda_vector = np.array([0.0,0.0,0.05])


# !!! Change this to the correct path on your computer!
npy_folder = "/Users/proden/Code/qed_CI-main/VLF_PQED/"

# these file names should still be good
E_npy_file = npy_folder + "LiH_r_1.4_6311g_fci_tight_davidson_Energies.npy"
Mu_npy_file = npy_folder + "LiH_r_1.4_6311g_fci_tight_davidson_Dipoles.npy"

# store energy eigenvalues in E_array
E_array = np.load(E_npy_file)
# store dipole matrix elements in Mu_array
Mu_array = np.load(Mu_npy_file)

# print their shape so we know how many elements we have
print(np.shape(E_array))
print(np.shape(Mu_array))

In [ ]:

N_ph = 2
omega = 0.12068

pqed = pQED(E_array, Mu_array)
energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=True)[0][0]
N_ph = 20
energy_pn_10_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

print(energy_pn_2_photons)
print(energy_cs_2_photons)

print(energy_pn_10_photons)


import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import pyplot as plt
import json
from matplotlib import cm
from matplotlib import rcParams


rcParams['font.family'] = 'serif'
rcParams['font.size'] = 12
# define color palette
COLOUR1 = "firebrick"
COLOUR2 = "green"
COLOUR3 = "royalblue"
COLOUR4 = "rebeccapurple"
COLOUR5 = 'darkorchid'
COLOUR6 = 'olivedrab'

COLOUR7 = 'violet'
COLOUR8 = 'yellow'


plt.axhline(energy_pn_2_photons, color=COLOUR3, label='energy_pn_2_photons')
plt.axhline(energy_cs_2_photons, color=COLOUR4, label='energy_cs_2_photons')
plt.axhline(energy_pn_10_photons, color=COLOUR5, label='energy_pn_10_photons')
plt.ylim(-8.01, -8.009)
plt.legend()
plt.show()



In [ ]:

import torch_optimizer as optim_
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
import copy
import tqdm

import torch.optim as optim


class vlf_PQED:

    def __init__(self, E_array, mu_array, LR = 0, num_epochs = 0):
        self.E_array = E_array
        self.mu_array = mu_array
        self.LR = LR
        self.num_epochs = num_epochs


    class CoherentStateTransform(nn.Module):
        
        def __init__(self, n_el, n_ph, omega, lambda_vector, E_array, mu_array, guess_vlf_params = None):
            super().__init__()
            self.n_el = n_el
            self.n_ph = n_ph
            self.omega = torch.tensor(omega, dtype=torch.double, requires_grad=True)
            self.lambda_vector = torch.tensor(lambda_vector, dtype=torch.double, requires_grad=True)
            self.E_array = torch.tensor(E_array[:n_el], dtype=torch.double, requires_grad=True)
            self.mu_array = torch.tensor(mu_array,  dtype=torch.double,requires_grad=True)
            self._d = torch.tensor(self.build_d_array(n_el, self.lambda_vector, self.mu_array), dtype=torch.double, requires_grad=True)

            # # Coherent state parameters for each state
            self.eta_k = nn.Parameter(-(1/(np.sqrt(omega*2))) * torch.ones(n_el, dtype=torch.double) *torch.diag(self._d), requires_grad=True)
            try:
                self.eta_k = nn.Parameter(torch.tensor(guess_vlf_params), requires_grad=True)
            except:
                print("couldn't use guess eta_k params")

            print("eta k beginning: ", self.eta_k)
            # self.f = nn.Parameter(torch.ones(1, dtype=torch.double)  * 0.75, requires_grad=True)

            #self.eta_k = nn.Parameter(-(1/(np.sqrt(omega*2))) * torch.ones(n_el, dtype=torch.double) *self._d[0,0], requires_grad=True)


        def forward(self):
            #return only eigvals
            vals, vecs , H =  self.PCQED_Hamiltonian_transformed()
            self.H = H
            return vals


        @staticmethod
        def laguerre_poly(n, k, z):
            """
            Vectorized computation of generalized Laguerre polynomials L_n^(k)(z).
            """
            n = n.to(dtype=torch.int64)
            k = k.to(dtype=torch.int64)
            z = z.to(dtype=torch.float64)

            L0 = torch.ones_like(z, dtype=torch.float64)
            L1 = 1 + k - z

            result = torch.where(n == 0, L0, torch.where(n == 1, L1, torch.zeros_like(z)))

            max_n = n.max().item()
            L_prev = L0
            L_curr = L1

            for i in range(2, max_n + 1):
                L_next = ((2 * i - 1 + k - z) * L_curr - (i - 1 + k) * L_prev) / i
                L_prev, L_curr = L_curr, L_next
                result = torch.where(n == i, L_curr, result)

            return result

        @staticmethod
        def factorial(n):
            """
            Compute the factorial of n using torch.lgamma for gradient support,
            with hard-coded handling for n = 0 and n = 1.

            Parameters:
            n (Tensor): Input tensor of non-negative integers.

            Returns:
            Tensor: Factorial of n.
            """
            # Ensure n is at least float for operations
            n = n.to(dtype=torch.float64)

            # Handle special cases manually
            out = torch.exp(torch.lgamma(n + 1.0))

            # Set factorial(0) = 1 and factorial(1) = 1 manually
            special_case = (n == 0) | (n == 1)
            out = torch.where(special_case, torch.ones_like(out), out)

            return out

        @staticmethod
        def displacement_matrix_element(n, m, z):
            """
            Compute ⟨n|e^(-z(b - b†))|m⟩ for arrays of n, m, and z.

            Parameters:
            n (Tensor): Tensor of non-negative integers.
            m (Tensor): Tensor of non-negative integers.
            z (Tensor): Displacement parameter.

            Returns:
            Tensor: Matrix elements ⟨n|e^(-z(b - b†))|m⟩.
            """

            n = n.to(dtype=torch.float64)
            m = m.to(dtype=torch.float64)
            z =- z.to(dtype=torch.float64)

            with torch.no_grad():

                abs_diff = torch.abs(n - m).to(dtype=torch.int64)
                min_nm = torch.minimum(n, m)
                max_nm = torch.maximum(n, m)

                # Compute factorial terms
                fact_min = vlf_PQED.CoherentStateTransform.factorial( min_nm)
                fact_max = vlf_PQED.CoherentStateTransform.factorial( max_nm)

                prefactor = torch.sqrt(fact_min / fact_max)

                # Create a mask for the condition n >= m
                n_ge_m_mask = (n >= m).to(dtype=torch.float64)
            
            # Compute power term using gradient-friendly masks
            # power_term_negative = (-z) ** abs_diff
            # power_term_positive = z ** abs_diff
                
            # power_term_negative = torch.sign(-z)**abs_diff * torch.abs(z)**abs_diff
            # power_term_positive = torch.sign(z)**abs_diff * torch.abs(z)**abs_diff

            eps = 1e-15
            power_term_negative = torch.exp(abs_diff * torch.log(torch.abs(z) + eps)) * torch.sign(-z)**abs_diff
            power_term_positive = torch.exp(abs_diff * torch.log(torch.abs(z) + eps)) *  torch.sign(z)**abs_diff


            power_term = n_ge_m_mask * power_term_positive + (1 - n_ge_m_mask) * power_term_negative


            # Compute exponential term
            exp_term = torch.exp(-0.5 * z ** 2)

            # Compute Laguerre polynomial

            laguerre = vlf_PQED.CoherentStateTransform.laguerre_poly(min_nm.to(dtype=torch.int64), abs_diff.to(dtype=torch.int64), z ** 2)

            return prefactor * power_term * exp_term * laguerre

        @staticmethod
        def displacement_matrix_element_b_dag_plus_b(n, m ,z):

            """
            Compute ⟨n|e^(-z(b - b†))  (b† + b)  |m⟩ for arrays of n, m, and z.

            Parameters:
            n (Tensor): Tensor of non-negative integers.
            m (Tensor): Tensor of non-negative integers.
            z (Tensor): Displacement parameter.

            Returns:
            Tensor: Matrix elements ⟨n|e^(-z(b - b†)) (b† + b) |m⟩.
            """
                        
            n = n.to(dtype=torch.float64)
            m = m.to(dtype=torch.float64)
            z = z.to(dtype=torch.float64)
            
            # Create a mask for valid m values (m >= 1)
            m_valid_mask = (m >= 1).to(dtype=torch.float64)
            
            # Term 1: √m ⟨n|e^(-z(b - b†))|m-1⟩
            # Only compute for valid m values
            m_minus_1 = torch.maximum(m - 1, torch.zeros_like(m))  # Ensure m-1 doesn't go negative
            term1 = torch.sqrt(m) * vlf_PQED.CoherentStateTransform.displacement_matrix_element(n, m_minus_1, z) * m_valid_mask
            
            # Term 2: √(m+1) ⟨n|e^(-z(b - b†))|m+1⟩
            term2 = torch.sqrt(m + 1) * vlf_PQED.CoherentStateTransform.displacement_matrix_element(n, m + 1, z)

            return term1 + term2


        @staticmethod
        def transform_one_body_VLF( h, lambda_values, n_ph):
            """
            Transform the Hamiltonian by incorporating photon displacement effects.
            Stores the one-body terms as a tensor of shape (n_ph, n_ph, n_orbs, n_orbs).
            
            Parameters:
            h (Tensor): One-body electronic Hamiltonian (n_orbs x n_orbs)
            lambda_values (Tensor): Array of lambda_p values (length n_orbs)
            n_ph (int): Maximum photon number state considered
            C: coefficient matrix (not used here)
            
            Returns:
            Tensor: Transformed one-body integrals stored as (n_ph x n_ph x n_orbs x n_orbs)
            """
            n_orbs = h.shape[0]

            # Compute the displacement matrix z = lambda_q - lambda_p
            lambda_p = lambda_values.view(-1, 1)  # Shape: (n_orbs, 1)
            lambda_q = lambda_values.view(1, -1)  # Shape: (1, n_orbs)
            #z = lambda_q - lambda_p  # Shape: (n_orbs, n_orbs)
            z = lambda_q - lambda_p  # Shape: (n_orbs, n_orbs)


            # print("lambda_values : ", lambda_values)
            # print("lambda_p: ", lambda_p)
            # print("lambda_q: ", lambda_q)
            # print("z : ,", z)

            # Vectorize the computation of displacement matrix elements D_nm
            # Create tensors for photon states n and m
            n = torch.arange(n_ph, device=h.device).view(n_ph, 1, 1, 1)  # Shape: (n_ph, 1, 1, 1)
            m = torch.arange(n_ph, device=h.device).view(1, n_ph, 1, 1)  # Shape: (1, n_ph, 1, 1)
            z_expanded = z.view(1, 1, n_orbs, n_orbs)  # Shape: (1, 1, n_orbs, n_orbs)

            # Compute D_nm for all combinations using a vectorized approach
            D_nm = vlf_PQED.CoherentStateTransform.displacement_matrix_element(n, m, z_expanded)  # Shape: (n_ph, n_ph, n_orbs, n_orbs)

            # Multiply h with D_nm, utilizing broadcasting
            h_expanded = h.view(1, 1, n_orbs, n_orbs)  # Shape: (1, 1, n_orbs, n_orbs)
            H_transformed = h_expanded * D_nm  # Shape: (n_ph, n_ph, n_orbs, n_orbs)

            return H_transformed


        @staticmethod
        def transform_hamiltonian_with_bdagplusb( h, lambda_values, n_ph):
            """
            Transform the Hamiltonian by incorporating photon displacement effects and b_dagger_plus_b.
            Stores the one-body terms as a tensor of shape (n_ph, n_ph, n_orbs, n_orbs).
            
            Parameters:
            h (Tensor): One-body electronic Hamiltonian (n_orbs x n_orbs)
            lambda_values (Tensor): Array of lambda_p values (length n_orbs)
            n_ph (int): Maximum photon number state considered
            C: coefficient matrix (not used here)
            
            Returns:
            Tensor: Transformed one-body integrals stored as (n_ph x n_ph x n_orbs x n_orbs)
            """
            n_orbs = h.shape[0]

            # Compute the displacement matrix z = lambda_q - lambda_p
            lambda_p = lambda_values.view(-1, 1)  # Shape: (n_orbs, 1)
            lambda_q = lambda_values.view(1, -1)  # Shape: (1, n_orbs)
            
            #z = lambda_q - lambda_p  # Shape: (n_orbs, n_orbs)
            z = lambda_q - lambda_p  # Shape: (n_orbs, n_orbs)

            # Vectorize the computation of displacement matrix elements D_nm
            # Create tensors for photon states n and m
            n = torch.arange(n_ph, device=h.device).view(n_ph, 1, 1, 1)  # Shape: (n_ph, 1, 1, 1)
            m = torch.arange(n_ph, device=h.device).view(1, n_ph, 1, 1)  # Shape: (1, n_ph, 1, 1)
            z_expanded = z.view(1, 1, n_orbs, n_orbs)  # Shape: (1, 1, n_orbs, n_orbs)

            # Compute D_nm for all combinations using a vectorized approach
            D_nm = vlf_PQED.CoherentStateTransform.displacement_matrix_element_b_dag_plus_b(n, m, z_expanded)  # Shape: (n_ph, n_ph, n_orbs, n_orbs)

            # Multiply h with D_nm, utilizing broadcasting
            h_expanded = h.view(1, 1, n_orbs, n_orbs)  # Shape: (1, 1, n_orbs, n_orbs)
            H_transformed = h_expanded * D_nm  # Shape: (n_ph, n_ph, n_orbs, n_orbs)

            return H_transformed

        @staticmethod
        def blockify_tensor(H_transformed):
            """
            Converts a 4D tensor of shape (n_ph, n_ph, n_els, n_els)
            into a 2D block matrix of shape (n_ph * n_els, n_ph * n_els)

            corresponds to kron( photonspace, electronicspace)
            """
            n_ph, _, n_orbs, _ = H_transformed.shape
            return H_transformed.permute(0, 2, 1, 3).reshape(n_ph * n_orbs, n_ph * n_orbs)




        def PCQED_Hamiltonian_transformed(self):
            """
            Build the PF Hamiltonian for a system with n_el electronic states and n_ph photon states.
            """

            """
            Given an array of n_el E_R values and an n_ph states with fundamental energy omega
            build the PF Hamiltonian

            n_el : int
                the number of electronic states (n_el = 1 means only ground-state)

            n_ph : int
                the number of photon occupation states (n_ph = 1 means only the |0> state)

            omega : float
                the photon frequency

            lambda_vector : numpy array of floats
                the lambda vector

            E_array : n_el np.array of floats
                the electronic energies

            mu_array : (n_el x n_el x 3) np.array of floats
                mu[i, j, k] is the kth cartesian component of the dipole moment expectation value between
                state i and state j


            """

            self.eta_k_matrix  = torch.diag(self.eta_k)
           #print("eta k ", self.eta_k_matrix)

            #print(self.eta_k_matrix)

            # Identity matrices for each subsystem
            I_matter = torch.eye(self.n_el)
            I_photon = torch.eye(self.n_ph)

            _d =  self.build_d_array(self.n_el, self.lambda_vector, self.mu_array)

            #create bosonic subspace operators
            b = self.create_annihilation_operator(self.n_ph)
            b_dagger = self.create_creation_operator(self.n_ph)


            b_transformed = torch.kron(b, I_matter) - torch.kron(I_photon, self.eta_k_matrix)
            b_daggger_transformed = torch.kron(b_dagger, I_matter) - torch.kron(I_photon, self.eta_k_matrix)

            E_matter = self.E_array* I_matter
            E = torch.kron( I_photon, E_matter)

            Cav =  self.omega *  b_daggger_transformed @ b_transformed 

            # BLC_matter = _d.clone()
            # BLC_photon = b_daggger_transformed + b_transformed
            # BLC_matter_transformed = vlf_PQED.CoherentStateTransform.blockify_tensor(vlf_PQED.CoherentStateTransform.transform_one_body_VLF(BLC_matter, self.eta_k, self.n_ph))
            # BLC_matter_transformed @ BLC_photon

            BLC_matter = _d.clone()
            BLC_photon = b_dagger+ b
            BLC_matter_transformed = vlf_PQED.CoherentStateTransform.blockify_tensor(vlf_PQED.CoherentStateTransform.transform_hamiltonian_with_bdagplusb(BLC_matter, self.eta_k, self.n_ph))


            BLC_matter_transformed = BLC_matter_transformed - 2* vlf_PQED.CoherentStateTransform.blockify_tensor(vlf_PQED.CoherentStateTransform.transform_one_body_VLF(BLC_matter @ self.eta_k_matrix, self.eta_k, self.n_ph))

            # Tensor product for combined subspace
            BLC =  -torch.sqrt(self.omega/2) * BLC_matter_transformed


            DSE_matter  = vlf_PQED.CoherentStateTransform.blockify_tensor(vlf_PQED.CoherentStateTransform.transform_one_body_VLF(( _d @ _d), self.eta_k, self.n_ph))
            DSE = 0.5 *  DSE_matter

            H = E + Cav + DSE  +BLC

            vals, vecs  = torch.linalg.eigh(H)

            return vals, vecs, H



        def return_params(self):
            return [self.eta_k]

        
        def build_d_array(self, n_el,lambda_vector,mu_array, ):
            """
            method to compute the array d = \lambda \cdot \mu if coherent_state==False
            or d = \lambda \cdot (\mu - <\mu>) if coherent_state == True
            and store to attribute .d_array

            """

            # print(type(lambda_vector))
            # print(type(mu_array))
            d_array = torch.einsum("k,ijk->ij", lambda_vector, mu_array[:n_el, :n_el, :])
            return d_array

        
        def create_annihilation_operator(self, N):
            """
            Creates the matrix representation of the annihilation operator (b) for a harmonic oscillator
            in a Hilbert space with N levels.
            
            Parameters:
            N (int): Number of levels.
            
            Returns:
            np.ndarray: The matrix representation of the annihilation operator.
            """
            b = torch.zeros((N, N))
            
            for j in range(1, N):
                b[j-1, j] = torch.sqrt(torch.tensor(j))
            
            return b

        def create_creation_operator(self, N):
            """
            Creates the matrix representation of the creation operator (b†) for a harmonic oscillator
            in a Hilbert space with N levels.
            
            Parameters:
            N (int): Number of levels.
            
            Returns:
            np.ndarray: The matrix representation of the creation operator.
            """
            b_dagger = torch.zeros((N, N))
            
            for j in range(1, N):
                b_dagger[j, j-1] = torch.sqrt(torch.tensor(j))
            
            return b_dagger



        def create_number_operator(self, N):
            """
            Creates the matrix representation of the number operator (n) = b† * b.
            
            Parameters:
            N (int): Number of levels.
            
            Returns:
            np.ndarray: The matrix representation of the number operator.
            """
            b = self.create_annihilation_operator(N)
            b_dagger = self.create_creation_operator(N)
            
            # The number operator is n = b† * b
            n = b_dagger @ b
            
            return n

        # def factorial(self, n):
        #     """Compute the factorial of n using PyTorch."""
        #     # Create a tensor of values [1, 2, ..., n] and compute the product
        #     return torch.cumprod(torch.arange(1, n + 1, dtype=torch.float32), dim=0)[-1]


    # Define a function to clamp the parameters
    def clamp_parameters(self, model, min_val, max_val):
        for param in model.parameters():
            param.data.clamp_(min_val, max_val)


    def PQED_Hamiltonian_vlf(  self, n_el,n_ph,omega,lambda_vector, number_to_minimize = 1, verbose =  False, guess_eta_k_params = None):

        self.model = self.CoherentStateTransform( n_el, n_ph, omega, lambda_vector, self.E_array, self.mu_array, guess_vlf_params=guess_eta_k_params)


        #number of epochs
        n_epochs =  self.num_epochs

        #learning rate
        LR = self.LR

        #LBFGS optimizer
        optimizer = torch.optim.LBFGS(self.model.return_params(), lr=LR, max_iter=100, max_eval=None, tolerance_grad=1e-20, tolerance_change=1e-20, history_size=250, line_search_fn='strong_wolfe')
        #optimizer = optim.Adadelta(self.model.return_params(), lr=LR, rho = 0.96)
        # Hold the best model
        best_loss = np.inf  # init to infinity
        best_weights = None
        history = []

        energies = []

        for epoch in range(n_epochs):
            running_loss = 0
            self.model.train()

            def closure():
                optimizer.zero_grad()
                y_pred = self.model()
                loss = torch.sum(y_pred[:number_to_minimize])

                #scaled_loss = loss * 10e6
                #loss.backward(retain_graph=True)

                grads = torch.autograd.grad(loss, self.model.parameters(), create_graph=True, allow_unused=True)
                #Manually set gradients for optimizer step
                for param, grad in zip(self.model.parameters(), grads):
                    param.grad = grad

                return loss


            optimizer.step(closure)
            loss = closure()
            running_loss += loss.item()

            # # Clamp the parameters after the update
            # self.clamp_parameters(self.model, min_val=-1.0, max_val=1.0)

            energies.append(self.model().detach().numpy())

            print("epoch-", epoch, "   loss_train------", loss.item())
            history.append(np.log(loss.detach().numpy()))  # Detach before converting to NumPy
            if loss < best_loss:
                best_loss = loss
                best_weights = copy.deepcopy(self.model.state_dict())


        # restore model and return best accuracy
        self.model.load_state_dict(best_weights)


        return np.array(energies)


    def PQED_Hamiltonian_vlf_hess(self, n_el, n_ph, omega, lambda_vector, number_to_minimize=1,
                        verbose=False, coherent_state=False):
        self.model = self.CoherentStateTransform(
            n_el, n_ph, omega, lambda_vector, self.E_array, self.mu_array
        )

        def get_flat_params():
            return torch.cat([p.detach().flatten() for p in self.model.parameters()]).numpy()

        def set_flat_params(flat_params):
            idx = 0
            for p in self.model.parameters():
                numel = p.numel()
                new_val = torch.tensor(flat_params[idx:idx+numel], dtype=p.dtype).reshape(p.shape)
                p.data = new_val.to(p.data)
                idx += numel

        def loss_fn(flat_params):
            set_flat_params(flat_params)
            y_pred = self.model()
            loss = torch.sum(y_pred[:number_to_minimize])
            return loss.item()

        def grad_fn(flat_params):
            set_flat_params(flat_params)
            y_pred = self.model()
            loss = torch.sum(y_pred[:number_to_minimize])
            grads = torch.autograd.grad(loss, self.model.parameters(), create_graph=True)
            flat_grad = torch.cat([g.contiguous().view(-1) for g in grads]).detach().numpy()
            return flat_grad

        def hess_fn(flat_params):
            set_flat_params(flat_params)
            y_pred = self.model()
            loss = torch.sum(y_pred[:number_to_minimize])

            grads = torch.autograd.grad(loss, self.model.parameters(), create_graph=True)
            flat_grad = torch.cat([g.contiguous().view(-1) for g in grads])
            hessian_rows = []

            for g in flat_grad:
                row_grads = torch.autograd.grad(g, self.model.parameters(), retain_graph=True)
                hessian_row = torch.cat([rg.contiguous().view(-1) for rg in row_grads])
                hessian_rows.append(hessian_row)

            hessian = torch.stack(hessian_rows).detach().numpy()
            print("hessian: ", hessian)
            return hessian

        x0 = get_flat_params()

        result = minimize(
            fun=loss_fn,
            x0=x0,
            jac=grad_fn,
            hess=hess_fn,
            method='bfgs',
            options={'disp': verbose},
            tol = 1e-15
        )

        # Restore the best model weights
        set_flat_params(result.x)

        # Final evaluation for energy tracking (not per-epoch anymore)
        final_energy = self.model().detach().numpy()
        return final_energy

    def eval(self):
        with torch.no_grad():
            vals, vecs, H = self.model.PCQED_Hamiltonian_transformed()

        return vals, vecs, H


In [ ]:

N_ph = 2
omega = 0.12068

pqed = pQED(E_array, Mu_array)
energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=True)[0][0]
N_ph = 20
energy_pn_10_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

N_ph = 2
#pqed = vlf_pQED(E_array, Mu_array, LR = 20, num_epochs=500)
pqed_2 = vlf_PQED(E_array, Mu_array, LR =1, num_epochs=3)
energy_vlf_2_photons = pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector)[0][0]


print(energy_vlf_2_photons)

print(energy_pn_2_photons)
print(energy_cs_2_photons)

print(energy_pn_10_photons)


import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import pyplot as plt
import json
from matplotlib import cm
from matplotlib import rcParams


rcParams['font.family'] = 'serif'
rcParams['font.size'] = 12
# define color palette
COLOUR1 = "firebrick"
COLOUR2 = "green"
COLOUR3 = "royalblue"
COLOUR4 = "rebeccapurple"
COLOUR5 = 'darkorchid'
COLOUR6 = 'olivedrab'

COLOUR7 = 'violet'
COLOUR8 = 'yellow'


# plt.axhline(energy_vlf_2_photons, color=COLOUR1, label='energy_vlf_2_photons')
plt.axhline(energy_pn_2_photons, color=COLOUR3, label='energy_pn_2_photons')
plt.axhline(energy_cs_2_photons, color=COLOUR4, label='energy_cs_2_photons')
plt.axhline(energy_pn_10_photons, color=COLOUR5, label='energy_pn_10_photons')
plt.axhline(energy_vlf_2_photons, color=COLOUR1, label='energy_vlf_2_photons')
plt.ylim(-8.01, -8.009)
plt.legend()
plt.show()

In [ ]:

N_ph = 20

omega = 0.12068

pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)
#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()

cisd_mol_e = vals[0].detach().numpy()
cisd_mol_e_2 = vals[1].detach().numpy()
cisd_mol_e_3 = vals[2].detach().numpy()
cisd_mol_e_4 = vals[3].detach().numpy()
print(cisd_mol_e)
print(cisd_mol_e_2)
print(cisd_mol_e_3)
print(cisd_mol_e_4)


fully_photon_converged_1 = cisd_mol_e
fully_photon_converged_2 = cisd_mol_e_2
fully_photon_converged_3 = cisd_mol_e_3
fully_photon_converged_4 = cisd_mol_e_4



state_1 = []
state_2 = []
state_3 = []
state_4 = []

state_1_pn = []
state_2_pn = []
state_3_pn = []
state_4_pn = []


state_1_vlf = []
state_2_vlf = []
state_3_vlf = []
state_4_vlf = []



photon_number_states = []

for i in range(2,7):
    photon_number_states.append(i)

    N_ph = i



    pqed = pQED(E_array, Mu_array)
    # energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
    # energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=True)

    print("hfadihjalsfdjlsadfj;sajfds")
    #vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()

    cisd_mol_e = vals[0].detach().numpy()
    cisd_mol_e_2 = vals[1].detach().numpy()
    cisd_mol_e_3 = vals[2].detach().numpy()
    cisd_mol_e_4 = vals[3].detach().numpy()

    state_1.append(cisd_mol_e)
    state_2.append(cisd_mol_e_2)
    state_3.append(cisd_mol_e_3)
    state_4.append(cisd_mol_e_4)


 
    pqed = pQED(E_array, Mu_array)
    # energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
    # energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=False)
    #vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()


    cisd_mol_e = vals[0].detach().numpy()
    cisd_mol_e_2 = vals[1].detach().numpy()
    cisd_mol_e_3 = vals[2].detach().numpy()
    cisd_mol_e_4 = vals[3].detach().numpy()


    state_1_pn.append(cisd_mol_e)
    state_2_pn.append(cisd_mol_e_2)
    state_3_pn.append(cisd_mol_e_3)
    state_4_pn.append(cisd_mol_e_4)




    pqed_2 = vlf_PQED(E_array, Mu_array, LR =0.5, num_epochs=2)

    if i == 2:
        #pqed_2.PQED_Hamiltonian_vlf_hess(N_el, N_ph, omega, lambda_vector, number_to_minimize=4)
        pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector, number_to_minimize=4)
    else:
       pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector, number_to_minimize=4, guess_eta_k_params=guess_eta_k_params)
       #pqed_2.PQED_Hamiltonian_vlf_hess(N_el, N_ph, omega, lambda_vector, number_to_minimize=4)




    guess_eta_k_params = pqed_2.model.eta_k.detach().numpy()

    print("vkf params: ", pqed_2.model.eta_k)

    vals, vecs, H_CI = pqed_2.model.PCQED_Hamiltonian_transformed()

    cisd_mol_e = vals[0].detach().numpy()
    cisd_mol_e_2 = vals[1].detach().numpy()
    cisd_mol_e_3 = vals[2].detach().numpy()
    cisd_mol_e_4 = vals[3].detach().numpy()


    state_1_vlf.append(cisd_mol_e)
    state_2_vlf.append(cisd_mol_e_2)
    state_3_vlf.append(cisd_mol_e_3)
    state_4_vlf.append(cisd_mol_e_4)




    print("next")



state_1 = np.array(state_1) - fully_photon_converged_1
state_2 = np.array(state_2)- fully_photon_converged_2
state_3 = np.array(state_3)- fully_photon_converged_3
state_4 = np.array(state_4)- fully_photon_converged_4

state_1_pn = np.array(state_1_pn) - fully_photon_converged_1
state_2_pn = np.array(state_2_pn)- fully_photon_converged_2
state_3_pn = np.array(state_3_pn)- fully_photon_converged_3
state_4_pn = np.array(state_4_pn)- fully_photon_converged_4


print(state_2_vlf)
print(fully_photon_converged_2)

print(state_3_vlf)
print(fully_photon_converged_3)
state_1_vlf = np.array(state_1_vlf) - fully_photon_converged_1
state_2_vlf = np.array(state_2_vlf)- fully_photon_converged_2
state_3_vlf = np.array(state_3_vlf)- fully_photon_converged_3
state_4_vlf = np.array(state_4_vlf)- fully_photon_converged_4


plt.plot(photon_number_states,state_1, label = "state_1", color = 'red')
plt.plot(photon_number_states,state_2, label = "state_2", color = 'green')
plt.plot(photon_number_states,state_3, label = "state_3", color = 'blue')
plt.plot(photon_number_states,state_4, label = "state_4", color = 'orange')

plt.plot(photon_number_states,state_1_pn, label = "state_1", color = 'red', linestyle = 'dotted')
plt.plot(photon_number_states,state_2_pn, label = "state_2", color = 'green', linestyle = 'dotted')
plt.plot(photon_number_states,state_3_pn, label = "state_3", color = 'blue', linestyle = 'dotted')
plt.plot(photon_number_states,state_4_pn, label = "state_4", color = 'orange', linestyle = 'dotted')


plt.plot(photon_number_states,state_1_vlf, label = "state_1", color = 'red', linestyle = 'dashed')
plt.plot(photon_number_states,state_2_vlf, label = "state_2", color = 'green', linestyle = 'dashed')
plt.plot(photon_number_states,state_3_vlf, label = "state_3", color = 'blue', linestyle = 'dashed')
plt.plot(photon_number_states,state_4_vlf, label = "state_4", color = 'orange', linestyle = 'dashed')

plt.legend()
plt.yscale("log")
plt.xlabel("Number of photon basis states")
plt.ylabel("Energy Convergence")
plt.title("Energy Convergence of CS and VLF Approaches vs Number of Photon Basis States, for HHe+ FCI sto-3g ground state")
plt.show()





In [ ]:
print(state_2_vlf)

In [ ]:


import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

# First subplot: state_1 and state_1_vlf
axs[0, 0].plot(photon_number_states, state_1, label="CS", color='green')
axs[0, 0].plot(photon_number_states, state_1_pn, label="PN", color='red', linestyle='dotted')
axs[0, 0].plot(photon_number_states, state_1_vlf, label="VLF", color='blue', linestyle='dashed')
axs[0,0].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[0, 0].set_title("State 1")
axs[0, 0].set_yscale("log")
axs[0, 0].legend()

# Second subplot: state_2 and state_2_vlf
axs[0, 1].plot(photon_number_states, state_2, label="CS", color='green')
axs[0, 1].plot(photon_number_states, state_2_pn, label="PN", color='red', linestyle='dotted')
axs[0, 1].plot(photon_number_states, state_2_vlf, label="VLF", color='blue', linestyle='dashed')
axs[0, 1].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[0, 1].set_title("State 2")
axs[0, 1].set_yscale("log")
axs[0, 1].legend()

# Third subplot: state_3 and state_3_vlf
axs[1, 0].plot(photon_number_states, state_3, label="CS", color='green')
axs[1, 0].plot(photon_number_states, state_3_pn, label="PN", color='red', linestyle='dotted')
axs[1, 0].plot(photon_number_states, state_3_vlf, label="VLF", color='blue', linestyle='dashed')
axs[1, 0].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[1, 0].set_title("State 3")
axs[1, 0].set_yscale("log")
axs[1, 0].legend()

# Fourth subplot: state_4 and state_4_vlf
axs[1, 1].plot(photon_number_states, state_4, label="CS", color='green')
axs[1, 1].plot(photon_number_states, state_4_pn, label="PN", color='red', linestyle='dotted')
axs[1, 1].plot(photon_number_states, state_4_vlf, label="VLF", color='blue', linestyle='dashed')
axs[1, 1].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[1, 1].set_title("State 4")
axs[1, 1].set_yscale("log")
axs[1, 1].legend()

# Common labels
for ax in axs[1, :]:
    ax.set_xlabel("Number of photon basis states")
for ax in axs[:, 0]:
    ax.set_ylabel("Energy Convergence")

xticks =photon_number_states  # or use np.arange(...) for more control
for ax in axs.flat:
    ax.set_xticks(xticks)


#fig.suptitle("Energy Convergence vs Number of Photon Basis States\nfor LiH CAS(4,4) sto-3g", fontsize=14)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:

N_ph = 20
omega = 0.12068

pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)
#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()

cisd_mol_e = vals[0].detach().numpy()
cisd_mol_e_2 = vals[1].detach().numpy()
cisd_mol_e_3 = vals[2].detach().numpy()
cisd_mol_e_4 = vals[3].detach().numpy()
print(cisd_mol_e)
print(cisd_mol_e_2)
print(cisd_mol_e_3)
print(cisd_mol_e_4)


fully_photon_converged_1 = cisd_mol_e
fully_photon_converged_2 = cisd_mol_e_2
fully_photon_converged_3 = cisd_mol_e_3
fully_photon_converged_4 = cisd_mol_e_4



state_1 = []
state_2 = []
state_3 = []
state_4 = []

state_1_pn = []
state_2_pn = []
state_3_pn = []
state_4_pn = []


state_1_vlf = []
state_2_vlf = []
state_3_vlf = []
state_4_vlf = []



photon_number_states = []

for i in range(2,7):
    photon_number_states.append(i)

    N_ph = i



    pqed = pQED(E_array, Mu_array)
    # energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
    # energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=True)

    print("hfadihjalsfdjlsadfj;sajfds")
    #vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()

    cisd_mol_e = vals[0].detach().numpy()
    cisd_mol_e_2 = vals[1].detach().numpy()
    cisd_mol_e_3 = vals[2].detach().numpy()
    cisd_mol_e_4 = vals[3].detach().numpy()

    state_1.append(cisd_mol_e)
    state_2.append(cisd_mol_e_2)
    state_3.append(cisd_mol_e_3)
    state_4.append(cisd_mol_e_4)


 
    pqed = pQED(E_array, Mu_array)
    # energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
    # energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=False)
    #vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()

    cisd_mol_e = vals[0].detach().numpy()
    cisd_mol_e_2 = vals[1].detach().numpy()
    cisd_mol_e_3 = vals[2].detach().numpy()
    cisd_mol_e_4 = vals[3].detach().numpy()


    state_1_pn.append(cisd_mol_e)
    state_2_pn.append(cisd_mol_e_2)
    state_3_pn.append(cisd_mol_e_3)
    state_4_pn.append(cisd_mol_e_4)


    if i == 2:
        pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector, number_to_minimize=1)
    else:
        pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector, number_to_minimize=1, guess_eta_k_params=guess_eta_k_params)


    guess_eta_k_params = pqed_2.model.eta_k.detach().numpy()

    print("vkf params: ", pqed_2.model.eta_k)

    vals, vecs, H_CI = pqed_2.model.PCQED_Hamiltonian_transformed()

    cisd_mol_e = vals[0].detach().numpy()
    cisd_mol_e_2 = vals[1].detach().numpy()
    cisd_mol_e_3 = vals[2].detach().numpy()
    cisd_mol_e_4 = vals[3].detach().numpy()


    state_1_vlf.append(cisd_mol_e)
    state_2_vlf.append(cisd_mol_e_2)
    state_3_vlf.append(cisd_mol_e_3)
    state_4_vlf.append(cisd_mol_e_4)




    print("next")



state_1 = np.array(state_1) - fully_photon_converged_1
state_2 = np.array(state_2)- fully_photon_converged_2
state_3 = np.array(state_3)- fully_photon_converged_3
state_4 = np.array(state_4)- fully_photon_converged_4

state_1_pn = np.array(state_1_pn) - fully_photon_converged_1
state_2_pn = np.array(state_2_pn)- fully_photon_converged_2
state_3_pn = np.array(state_3_pn)- fully_photon_converged_3
state_4_pn = np.array(state_4_pn)- fully_photon_converged_4


print(state_2_vlf)
print(fully_photon_converged_2)

print(state_3_vlf)
print(fully_photon_converged_3)
state_1_vlf = np.array(state_1_vlf) - fully_photon_converged_1
state_2_vlf = np.array(state_2_vlf)- fully_photon_converged_2
state_3_vlf = np.array(state_3_vlf)- fully_photon_converged_3
state_4_vlf = np.array(state_4_vlf)- fully_photon_converged_4


plt.plot(photon_number_states,state_1, label = "state_1", color = 'red')
plt.plot(photon_number_states,state_2, label = "state_2", color = 'green')
plt.plot(photon_number_states,state_3, label = "state_3", color = 'blue')
plt.plot(photon_number_states,state_4, label = "state_4", color = 'orange')

plt.plot(photon_number_states,state_1_pn, label = "state_1", color = 'red', linestyle = 'dotted')
plt.plot(photon_number_states,state_2_pn, label = "state_2", color = 'green', linestyle = 'dotted')
plt.plot(photon_number_states,state_3_pn, label = "state_3", color = 'blue', linestyle = 'dotted')
plt.plot(photon_number_states,state_4_pn, label = "state_4", color = 'orange', linestyle = 'dotted')


plt.plot(photon_number_states,state_1_vlf, label = "state_1", color = 'red', linestyle = 'dashed')
plt.plot(photon_number_states,state_2_vlf, label = "state_2", color = 'green', linestyle = 'dashed')
plt.plot(photon_number_states,state_3_vlf, label = "state_3", color = 'blue', linestyle = 'dashed')
plt.plot(photon_number_states,state_4_vlf, label = "state_4", color = 'orange', linestyle = 'dashed')

plt.legend()
plt.yscale("log")
plt.xlabel("Number of photon basis states")
plt.ylabel("Energy Convergence")
plt.title("Energy Convergence of CS and VLF Approaches vs Number of Photon Basis States, for HHe+ FCI sto-3g ground state")
plt.show()



In [ ]:


import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

# First subplot: state_1 and state_1_vlf
axs[0, 0].plot(photon_number_states, state_1, label="CS", color='green')
axs[0, 0].plot(photon_number_states, state_1_pn, label="PN", color='red', linestyle='dotted')
axs[0, 0].plot(photon_number_states, state_1_vlf, label="VLF", color='blue', linestyle='dashed')
#axs[0,0].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[0, 0].set_title("State 1")
axs[0, 0].set_yscale("log")
axs[0, 0].legend()

# Second subplot: state_2 and state_2_vlf
axs[0, 1].plot(photon_number_states, state_2, label="CS", color='green')
axs[0, 1].plot(photon_number_states, state_2_pn, label="PN", color='red', linestyle='dotted')
axs[0, 1].plot(photon_number_states, state_2_vlf, label="VLF", color='blue', linestyle='dashed')
#axs[0, 1].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[0, 1].set_title("State 2")
axs[0, 1].set_yscale("log")
axs[0, 1].legend()

# Third subplot: state_3 and state_3_vlf
axs[1, 0].plot(photon_number_states, state_3, label="CS", color='green')
axs[1, 0].plot(photon_number_states, state_3_pn, label="PN", color='red', linestyle='dotted')
axs[1, 0].plot(photon_number_states, state_3_vlf, label="VLF", color='blue', linestyle='dashed')
#axs[1, 0].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[1, 0].set_title("State 3")
axs[1, 0].set_yscale("log")
axs[1, 0].legend()

# Fourth subplot: state_4 and state_4_vlf
axs[1, 1].plot(photon_number_states, state_4, label="CS", color='green')
axs[1, 1].plot(photon_number_states, state_4_pn, label="PN", color='red', linestyle='dotted')
axs[1, 1].plot(photon_number_states, state_4_vlf, label="VLF", color='blue', linestyle='dashed')
#axs[1, 1].axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs[1, 1].set_title("State 4")
axs[1, 1].set_yscale("log")
axs[1, 1].legend()

# Common labels
for ax in axs[1, :]:
    ax.set_xlabel("Number of photon basis states")
for ax in axs[:, 0]:
    ax.set_ylabel("Energy Convergence")

xticks =photon_number_states  # or use np.arange(...) for more control
for ax in axs.flat:
    ax.set_xticks(xticks)


#fig.suptitle("Energy Convergence vs Number of Photon Basis States\nfor LiH CAS(4,4) sto-3g", fontsize=14)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()



import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 1, figsize=(12, 8), sharex=True, sharey=True)

# First subplot: state_1 and state_1_vlf
axs.plot(photon_number_states, state_1, label="CS", color='green')
axs.plot(photon_number_states, state_1_pn, label="PN", color='red', linestyle='dotted')
axs.plot(photon_number_states, state_1_vlf, label="VLF", color='blue', linestyle='dashed')
##axs.axhline(y=1e-9, color='purple', linestyle='--', linewidth=2)
axs.set_title("State 1")
axs.set_yscale("log")
axs.legend()



axs.set_xlabel("Number of photon basis states")

axs.set_ylabel("Energy Convergence")

xticks =photon_number_states  # or use np.arange(...) for more control

axs.set_xticks(xticks)


#fig.suptitle("Energy Convergence vs Number of Photon Basis States\nfor LiH CAS(4,4) sto-3g", fontsize=14)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use seaborn color palette for better aesthetics
colors = sns.color_palette("colorblind", 3)  # CS, PN, VLF

# Define common marker styles for each line
markers = ['o', 's', 'D']

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

# List of data for cleaner looping
state_data = [
    (state_1, state_1_pn, state_1_vlf, "State 1"),
    (state_2, state_2_pn, state_2_vlf, "State 2"),
    (state_3, state_3_pn, state_3_vlf, "State 3"),
    (state_4, state_4_pn, state_4_vlf, "State 4"),
]

for idx, ax in enumerate(axs.flat):
    cs, pn, vlf, title = state_data[idx]
    ax.plot(photon_number_states, cs, label="CS", color=colors[0], marker=markers[0])
    ax.plot(photon_number_states, pn, label="PN", color=colors[1], marker=markers[1])
    ax.plot(photon_number_states, vlf, label="VLF", color=colors[2], marker=markers[2])
    ax.set_title(title, fontsize=14)
    ax.set_yscale("log")
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.grid(True, linestyle='--', linewidth=0.5)

# Common axis labels
for ax in axs[1, :]:
    ax.set_xlabel("Number of Photon Basis States", fontsize=12)
for ax in axs[:, 0]:
    ax.set_ylabel("Energy Convergence", fontsize=12)

# Set uniform xticks
xticks = photon_number_states
for ax in axs.flat:
    ax.set_xticks(xticks)

# Put legend only once in the top-right plot
axs[0, 1].legend(fontsize=11, loc='best')

# Improve layout
fig.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(8, 6))

# Plot all three with markers and solid lines
ax.plot(photon_number_states, state_1, label="CS", color=colors[0], marker=markers[0])
ax.plot(photon_number_states, state_1_pn, label="PN", color=colors[1], marker=markers[1])
ax.plot(photon_number_states, state_1_vlf, label="VLF", color=colors[2], marker=markers[2])

#ax.set_title("State 1", fontsize=14)
ax.set_yscale("log")
ax.set_xlabel("Number of Photon Basis States", fontsize=12)
ax.set_ylabel("Energy Convergence", fontsize=12)
ax.set_xticks(photon_number_states)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.grid(True, linestyle='--', linewidth=0.5)
ax.legend(fontsize=11, loc='best')

fig.tight_layout()
plt.show()


In [ ]:


def create_annihilation_operator(N):
    """
    Creates the matrix representation of the annihilation operator (b) for a harmonic oscillator
    in a Hilbert space with N levels.
    
    Parameters:
    N (int): Number of levels.
    
    Returns:
    np.ndarray: The matrix representation of the annihilation operator.
    """
    b = np.zeros((N, N))
    
    for j in range(1, N):
        b[j-1, j] = np.sqrt(j)
    
    return b

def create_creation_operator(N):
    """
    Creates the matrix representation of the creation operator (b†) for a harmonic oscillator
    in a Hilbert space with N levels.
    
    Parameters:
    N (int): Number of levels.
    
    Returns:
    np.ndarray: The matrix representation of the creation operator.
    """
    b_dagger = np.zeros((N, N))
    
    for j in range(1, N):
        b_dagger[j, j-1] = np.sqrt(j)
    
    return b_dagger

    
def compute_energy_corrections(order, H0, V ):
    n_max = H0.shape[0]
    n = 0

    # Diagonalize the unperturbed Hamiltonian
    E0, psi0 = np.linalg.eigh(H0)



    #psi0[:, 0] is firt wavefunction
    #print(psi0.shape)

    # Initialize corrections to energy and wavefunction
    energy_corrections = np.zeros(( order+1))
    wavefunction_corrections = np.zeros((n_max, order+1))



    # Set unperturbed energies and wavefunctions
    energy_corrections[ 0] = E0[n]
    wavefunction_corrections[ :, 0] = psi0[:, n]


    n_0 = 0


    for k in range(1, order + 1):
        
        if k == 1:
            #first order energy correction
            energy_corrections[1] = np.dot(psi0[:, n].T, np.dot(V, psi0[:,n]))

            #print(energy_corrections[1])


            coeff_m = 0
            for m in range(0, n_max):
                coeff_m = 0
                if m!= n:


                    coeff_m -=   np.dot(psi0[:, m].T, np.dot(V, psi0[:,n]))

                    for l in range(1, k+1):
                        #goes from l to k-1
                        coeff_m += energy_corrections[l] * np.dot(psi0[:, m].T, wavefunction_corrections[:, k-l])

                    coeff_m = coeff_m/np.abs(E0[m] - E0[n])

                    wavefunction_corrections[:, k ] += (coeff_m * psi0[:,m])


        if k!=1:  
            for m in range(0, n_max):
                coeff_m = 0
                if m!= n:
                    coeff_m -=  np.dot(psi0[:,m].T, np.dot(V, wavefunction_corrections[:, k-1]))
                    for l in range(1, k+1):
                        #goes from l to k-1
                        coeff_m += energy_corrections[l] * np.dot(psi0[:, m].T, wavefunction_corrections[:, k-l])

                    coeff_m = coeff_m/np.abs(E0[m] - E0[n])
                    # print("coeff_ m for state ", m , ": ", coeff_m)

                    wavefunction_corrections[:, k ] += (coeff_m * psi0[:,m])



        if k!= 1:
            energy_correction =  np.dot(psi0[:,n].T, np.dot(V, wavefunction_corrections[:, k-1]))
            
            for j in range(0, k):
                if j != k:
                    #print("energy correction for ", j ," : " ,energy_correction)
                    energy_correction -=  energy_corrections[j]  * np.dot(psi0[:,n].T,  wavefunction_corrections[:, k-j])

            energy_corrections[ k] = energy_correction


    #print("energy_corrections: " , energy_corrections)
    return energy_corrections



# #tesign this real quick
# H0 = np.array([[1.0, 0.0], [0.0, 2.0]])
# V = np.array([[0.0, 0.01], [0.01, 0.0]])
# corrs = compute_energy_corrections(30, H0, V)
# print(corrs)
# print(np.sum(corrs))
# vals, vecs = np.linalg.eigh(H0+V)
# print(vals)



H = pqed_2.model.H.detach().numpy()
print(H.shape)

vals,vecs = np.linalg.eigh(H)
print(vals[0])

#build H0 subtract it from full H  just H_e and H_cav
n_el = int(H.shape[0]/N_ph)
n_ph = int(H.shape[0]/N_el)

b = create_annihilation_operator(n_ph)
b_dag = create_creation_operator(n_ph)
H0 = np.kron(np.eye(n_ph), np.diag( E_array[:n_el])) + omega * np.kron(b_dag @ b, np.eye(n_el))
V = H- H0

print(V)

corrs = compute_energy_corrections(30, H0, V)
print(np.sum(corrs))


In [ ]:

#converged
N_ph = 15
pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=False)

#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
print("vals 0 :", vals[0].detach().numpy())

converged_ground = vals[0].detach().numpy()




order = 10
N_ph = 10

pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=True)


#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
print("vals 0 :", vals[0].detach().numpy())


H=pqed.model.H.detach().numpy()


#build H0 subtract it from full H  just H_e and H_cav
n_el = int(H.shape[0]/N_ph)
n_ph = int(H.shape[0]/N_el)
print("n_el: ", n_el)

b = create_annihilation_operator(n_ph)
b_dag = create_creation_operator(n_ph)
H0 = np.kron(np.eye(n_ph), np.diag( E_array[:n_el])) + omega * np.kron(b_dag @ b, np.eye(n_el))


V = H- H0
corrs_CS = compute_energy_corrections(order, H0, V)



pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=False)
#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
print("vals 0 :", vals[0].detach().numpy())


H=pqed.model.H.detach().numpy()
V = H- H0
corrs_PN = compute_energy_corrections(order, H0, V)




pqed_2 = vlf_PQED(E_array, Mu_array, LR =1, num_epochs=2)
pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector, number_to_minimize=1)

vals, vecs, H_CI = pqed_2.model.PCQED_Hamiltonian_transformed()

print("vals 0 :", vals[0].detach().numpy())

H=pqed_2.model.H.detach().numpy()
V = H- H0
corrs_VLF = compute_energy_corrections(order, H0, V)


print(np.sum(corrs_PN))
print(np.sum(corrs_CS))
print(np.sum(corrs_VLF))







In [ ]:

corrected_es_pn = []
corrected_es_cs = []
corrected_es_vlf = []

for i in range(0, order):
    corrected_es_cs.append(np.sum(corrs_CS[:i+1]))
    corrected_es_pn.append(np.sum(corrs_PN[:i+1]))
    corrected_es_vlf.append( np.sum(corrs_VLF[:i+1]))

corrected_es_pn = converged_ground -  np.array(corrected_es_pn) 
corrected_es_cs = converged_ground -  np.array(corrected_es_cs) 
corrected_es_vlf = converged_ground -  np.array(corrected_es_vlf) 

print(corrected_es_vlf)
print(corrected_es_cs)
print(corrected_es_pn)


import numpy as np
import matplotlib.pyplot as plt

# Define custom color palette
colors = {
    'vlf': '#1f77b4',  # blue
    'cs': '#ff7f0e',   # orange
    'pn': '#2ca02c'    # green
}

# Create the plot
plt.figure(figsize=(10, 6))  # Slightly larger figure for clarity

# Plot each series with specified color and line style
plt.plot(corrected_es_vlf, label='VLF', color=colors['vlf'], linewidth=2, marker='o',  markersize=12, fillstyle='none', markeredgewidth=2)
plt.plot(corrected_es_cs, label='CS', color=colors['cs'], linewidth=2, marker='s', markersize=12, fillstyle='none', markeredgewidth=2)
plt.plot(corrected_es_pn, label='PN', color=colors['pn'], linewidth=2, marker='^', markersize=12, fillstyle='none', markeredgewidth=2)

# Set y-axis to symmetric log scale
plt.yscale('symlog', linthresh=1e-8)

# Add titles and labels with improved font size
#plt.title('', fontsize=14, fontweight='bold')
plt.xlabel('Perturbation Theory Order', fontsize=12)
plt.ylabel('Energy Error (Hartree)', fontsize=12)

# Enable grid
plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# Add legend with custom styling
plt.legend(frameon=True, framealpha=0.9, fontsize=11)

# Optional: Improve layout
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

def plot_array_magnitude(matrix):
    """
    Plots the magnitude of a 2D NumPy array with logarithmic color scaling in a publication-quality style.
    """
    magnitude = np.abs(matrix)
    magnitude[magnitude == 0] = 1e-15  # Prevent log(0)

    fig, ax = plt.subplots(figsize=(8, 6))

    cax = ax.imshow(
        magnitude,
        cmap='nipy_spectral',  # Better for perceptual uniformity; 'viridis' or 'inferno' also great
        norm=LogNorm(vmin=1e-15, vmax=magnitude.max()),
        interpolation='nearest',
        aspect='auto'
    )

    # Colorbar with tight styling
    cbar = fig.colorbar(cax, ax=ax, pad=0.02, fraction=0.046)
    cbar.set_label(r'$\log_{10}(\mathrm{Magnitude})$', fontsize=13)
    cbar.ax.tick_params(labelsize=11)

    # Axis positioning
    ax.xaxis.set_ticks_position('top')
    ax.xaxis.set_label_position('top')
    ax.set_xlabel('Column Index', fontsize=13)
    ax.set_ylabel('Row Index', fontsize=13)

    # Tick styling
    ax.tick_params(axis='both', which='major', labelsize=11, direction='in')

    # Grid for reference (optional)
    ax.grid(False)

    # Tight layout for clean spacing
    plt.tight_layout()
    plt.show()



order = 8
N_ph = 7
N_el_plot_matrix = 10

pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
pqed.PQED_Hamiltonian(N_el_plot_matrix, N_ph, omega, lambda_vector, coherent_state=True)


#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
print("vals 0 :", vals[0].detach().numpy())


H=pqed.model.H.detach().numpy()

plot_array_magnitude(H)


#build H0 subtract it from full H  just H_e and H_cav
n_el = int(H.shape[0]/N_ph)
n_ph = int(H.shape[0]/N_el_plot_matrix)

b = create_annihilation_operator(n_ph)
b_dag = create_creation_operator(n_ph)
H0 = np.kron(np.eye(n_ph), np.diag( E_array[:n_el])) + omega * np.kron(b_dag @ b, np.eye(n_el))


V = H- H0
# corrs_CS = compute_energy_corrections(order, H0, V)



pqed = pQED(E_array, Mu_array)
# energy_pn_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]
# energy_cs_2_photons = pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)[0][0]

pqed.PQED_Hamiltonian(N_el_plot_matrix, N_ph, omega, lambda_vector, coherent_state=False)
#vals, vecs, H_CI = qedci.calculate_CI_energy(excitation_level=2, num_active_electrons=2, num_active_orbitals=4)
vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
print("vals 0 :", vals[0].detach().numpy())


H=pqed.model.H.detach().numpy()

plot_array_magnitude(H)

V = H- H0
# corrs_PN = compute_energy_corrections(order, H0, V)




pqed_2 = vlf_PQED(E_array, Mu_array, LR =0.2, num_epochs=2)
pqed_2.PQED_Hamiltonian_vlf(N_el_plot_matrix, N_ph, omega, lambda_vector, number_to_minimize=1)

vals, vecs, H_CI = pqed_2.model.PCQED_Hamiltonian_transformed()

print("vkf params: ", pqed_2.model.eta_k)

print("vals 0 :", vals[0].detach().numpy())

H=pqed_2.model.H.detach().numpy()

plot_array_magnitude(H)

V = H- H0
# corrs_VLF = compute_energy_corrections(order, H0, V)

In [ ]:
N_R = 25
d_array = np.linspace(1.4, 2.2, N_R)




N_el = 50
N_ph = 2
omega = 0.12068
lambda_vector = np.array([0.0,0.0,0.035])


# !!! Change this to the correct path on your computer!
npy_folder = "/Users/proden/Code/POLARITONPERTURBATIONTHEORY/data/"

# these file names should still be good
E_npy_file = npy_folder + "LiH_r_scan_6311g_fci_tight_davidson_Energies.npy"
Mu_npy_file = npy_folder + "LiH_r_scan_6311g_fci_tight_davidson_Dipoles.npy"

# store energy eigenvalues in E_array
E_array = np.load(E_npy_file)
# store dipole matrix elements in Mu_array
Mu_array = np.load(Mu_npy_file)

# print their shape so we know how many elements we have
print(np.shape(E_array))
print(np.shape(Mu_array))
# print(E_array)

print(E_array[: 5, :].T.tolist()   )
print( (E_array[:5, :].T + omega).tolist())

import matplotlib.pyplot as plt
plt.plot(   np.array(E_array[: 5, :].tolist() +  (E_array[:5, :] + omega).tolist() ).T  , linestyle = 'dotted' )


N_ph = 2
omega= 0.12068

vals_for_plot_cs = []
vals_for_plot_pn = []
vals_for_plot_vlf = []
for i in range(0, E_array.shape[1]):
    e_ar = E_array[:, i]
    mu_ar = Mu_array[:,:,:, i]
    pqed = pQED(e_ar, mu_ar)
    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector, coherent_state=True)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
    vals_for_plot_cs.append(vals.detach().numpy())

    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
    vals_for_plot_pn.append(vals.detach().numpy())


    pqed_2 = vlf_PQED(e_ar, mu_ar, LR =0.5, num_epochs=2)
    pqed_2.PQED_Hamiltonian_vlf(N_el, N_ph, omega, lambda_vector, number_to_minimize=4)
    vals, vecs, H_CI = pqed_2.model.PCQED_Hamiltonian_transformed()
    vals_for_plot_vlf.append(vals.detach().numpy())

#plt.plot(np.array(vals_for_plot)[:, :5])

N_ph = 10

vals_for_plot_conv = []
for i in range(0, E_array.shape[1]):
    e_ar = E_array[:, i]
    mu_ar = Mu_array[:,:,:, i]
    pqed = pQED(e_ar, mu_ar)
    pqed.PQED_Hamiltonian(N_el, N_ph, omega, lambda_vector)
    vals, vecs, H_CI = pqed.model.PCQED_Hamiltonian_transformed()
    vals_for_plot_conv.append(vals.detach().numpy())






plt.plot(np.array(vals_for_plot_cs)[:, :5])
plt.plot(np.array(vals_for_plot_pn)[:, :5])
plt.plot(np.array(vals_for_plot_conv)[:, :5])





start_plot = 0
end_plot =  15

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))  # Increase figure size for clarity

# ---- Plot groups with colors, linestyles, and open markers ----
# PNCONV (black)
plt.plot(d_array[start_plot:end_plot] , np.array(vals_for_plot_conv)[:, 2][ start_plot:end_plot], color='black', linestyle='-', marker='^', markerfacecolor='white')
plt.plot(d_array[start_plot:end_plot] ,np.array(vals_for_plot_conv)[:, 3][ start_plot:end_plot], color='black', linestyle='-', marker='^', markerfacecolor='white', label='Converged')

# PN (red)
plt.plot( d_array[start_plot:end_plot] ,np.array(vals_for_plot_pn)[:, 2][ start_plot:end_plot], color='red', linestyle='-', marker='d', markerfacecolor='white')
plt.plot( d_array[start_plot:end_plot] ,np.array(vals_for_plot_pn)[:, 3][ start_plot:end_plot], color='red', linestyle='-', marker='d', markerfacecolor='white', label='PN')

# # VLF (green)
plt.plot(d_array[start_plot:end_plot] , np.array(vals_for_plot_vlf)[:, 2][start_plot:end_plot], color='green', linestyle='-', marker='o', markerfacecolor='white')
plt.plot(d_array[start_plot:end_plot] ,np.array(vals_for_plot_vlf)[:, 3][ start_plot:end_plot], color='green', linestyle='-', marker='o', markerfacecolor='white', label='VLF')

# Bare CI (blue)
plt.plot(d_array[start_plot:end_plot] , np.array(vals_for_plot_cs)[:, 2][start_plot:end_plot], color='blue', linestyle='-', marker='s', markerfacecolor='white', )
plt.plot(d_array[start_plot:end_plot] , np.array(vals_for_plot_cs)[:, 3][start_plot:end_plot], color='blue', linestyle='-', marker='s', markerfacecolor='white', label='CS')

# # No coupling (purple dashed)
# plt.plot(CI_ex_2_nocoup, color='purple', linestyle='--', marker='v', markerfacecolor='white', )
# plt.plot(CI_ex_3_nocoup, color='purple', linestyle='--', marker='v', markerfacecolor='white', label='No Coupling')

# ---- Aesthetics ----
plt.xlabel('Bond length (Å)', fontsize=12)
plt.ylabel('Energy ($E_\mathrm{h}$)', fontsize=12)
#plt.title('Comparison of CI Excitation Energies', fontsize=14)

plt.legend(fontsize=10, loc='best', frameon=False, ncol=2)  # adjust as needed
plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
plt.tight_layout()
plt.show()

